In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
df = pd.read_csv('../../data/cleaned.csv')
df

/var/folders/5g/sd7vmfvs2rn86tg601yfsjx80000gn/T/ipykernel_89555/1552466201.py:1: DtypeWarning: Columns (0: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../../data/cleaned.csv')


,ViewYN,PoolPrivateYN,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,LivingArea,DaysOnMarket,...,City_West Hills,City_Westminster,City_Whittier,City_Wildomar,City_Winchester,City_Winnetka,City_Woodland Hills,City_Yorba Linda,City_Yucaipa,City_Yucca Valley
0,0,0,1114319133,2025-05-30,554869.0,33.746904,-117.128111,25483 Calamity Lane,-0.075059,-0.818892,...,False,False,False,False,False,False,False,False,False,False
1,1,0,1114250894,2025-05-30,700000.0,33.479802,-117.069631,33686 Abbey Road,0.953256,-0.795849,...,False,False,False,False,False,False,False,False,False,False
2,1,0,1114246939,2025-05-28,875000.0,33.622840,-117.641489,28097 CALLE VALDES,-0.630871,-0.841935,...,False,False,False,False,False,False,False,False,False,False
3,1,0,1114212175,2025-05-28,870000.0,34.148868,-117.588025,5847 Zapata Place,0.294614,-0.772807,...,False,False,False,False,False,False,False,False,False,False
4,1,0,1113976206,2025-05-20,2430000.0,34.123876,-118.104931,2335 Cumberland Road,0.293312,-0.864978,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77512,1,1,1109015565,2026-05-04,2200000.0,33.844376,-117.557486,870 Encanto Street,4.025187,3.444019,...,False,False,False,False,False,False,False,False,False,False
77513,0,0,1108311252,2026-05-18,510000.0,34.073172,-117.726591,1441 Sheridan Avenue,-1.014862,-0.358037,...,False,False,False,False,False,False,False,False,False,False
77514,1,0,1088164599,2026-05-07,645000.0,33.632208,-117.224013,33680 Harvest Way E,1.415348,4.619200,...,False,False,False,True,False,False,False,False,False,False
77515,0,0,1087407440,2026-05-01,965000.0,33.879104,-118.213803,804 E Clemmer Drive,0.924620,-0.519336,...,False,False,False,False,False,False,False,False,False,False


In [11]:
cat_col = ['UnparsedAddress', 'AssociationFeeFrequency', 
        'MLSAreaMajor', 'ElementarySchool', 'SubdivisionName',  
        'PurchaseContractDate', 'MiddleOrJuniorSchool', 'HighSchool',
        'HighSchoolDistrict', 'PostalCode',  'ListingKey', 
        'ListingKeyNumeric', 'ListingId', 'ContractStatusChangeDate',
        'ListingContractDate' ]

In [12]:
df_model = df.drop(columns=cat_col)

In [13]:
# Convert CloseDate to datetime
df_model['CloseDate'] = pd.to_datetime(df_model['CloseDate'])
df_model = df_model.sort_values('CloseDate').reset_index(drop=True)

# Split dataframe into train and test sets using given window
def get_time_split(data, X_months):
    latest_date = data['CloseDate'].max()
    test_start_date = latest_date - pd.DateOffset(months=1)
    train_start_date = test_start_date - pd.DateOffset(months=X_months)
    test_set = data[data['CloseDate'] >= test_start_date]
    train_set = data[(data['CloseDate'] >= train_start_date) & (data['CloseDate'] < test_start_date)]
    return train_set, test_set

# Create testing and training sets, removing target and 'CloseDate'
train_df, test_df = get_time_split(df_model, X_months=6)
target = 'ClosePrice'
X_train = train_df.drop(columns=['ClosePrice', 'CloseDate'])
y_train = train_df[target]
X_test = test_df.drop(columns=['ClosePrice', 'CloseDate'])
y_test = test_df[target]

# When modeling, different testing windows will be evaulated
# window_options = [3, 6, 12]
# for X in window_options:
#     train_df, test_df = get_time_split(df, X_months=X)
    
#     print(f"--- Experimenting with X = {X} Months ---")
#     print(f"Train Date Range: {train_df['CloseDate'].min().strftime('%Y-%m-%d')} to {train_df['CloseDate'].max().strftime('%Y-%m-%d')} ({len(train_df)} rows)")
#     print(f"Test Date Range:  {test_df['CloseDate'].min().strftime('%Y-%m-%d')} to {test_df['CloseDate'].max().strftime('%Y-%m-%d')} ({len(test_df)} rows)\n")
    

In [17]:

print(f"Train:\n{train_df.shape}")
print(X_train.shape)
print(y_train.shape)
print(f"Test:\n{test_df.shape}")
print(X_test.shape)
print(y_test.shape)

Train:
(32801, 329)
(32801, 327)
(32801,)
Test:
(6544, 329)
(6544, 327)
(6544,)


In [15]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

In [16]:
from sklearn.metrics import r2_score, mean_squared_error

score = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print(f'R2 Value: {score}')
print(f'MSE Value: {mse}')

R2 Value: 0.8102743741523906
MSE Value: 64696894036.10693
